# 49 — Region-Constrained Design Search v3.1

**Thesis:** topology specifies robustness; robustness specifies operation; operation specifies computation.

This notebook implements a complete synthetic design-search pipeline:

\[
D \rightarrow \Omega(D) \rightarrow \Omega_c(D) \rightarrow d(\partial \Omega_c) \rightarrow x^* \rightarrow \text{fault-tolerant execution}.
\]

The specification is the **region**. The operating point is one admitted realization inside it.

**v3.1 upgrade:** adds a Colab/Jupyter-safe output packaging cell that automatically downloads the generated figures/data archive when possible, with a clickable fallback link.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Ellipse
from scipy import ndimage

OUT = Path("outputs/notebook_49_v3")
FIG = OUT / "figures"
DATA = OUT / "data"
FIG.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "font.size": 11,
    "axes.titlesize": 19,
    "axes.labelsize": 12,
})

def savefig(fig, name):
    path = FIG / name
    fig.savefig(path, bbox_inches="tight")
    return path

def clean_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

def draw_box(ax, xy, wh, title, subtitle="", lw=2.0, title_size=11, subtitle_size=8):
    x, y = xy
    w, h = wh
    ax.add_patch(Rectangle((x, y), w, h, fill=False, linewidth=lw, edgecolor="black"))
    ax.text(x+w/2, y+h*0.66, title, ha="center", va="center",
            fontsize=title_size, fontweight="bold")
    if subtitle:
        ax.text(x+w/2, y+h*0.28, subtitle, ha="center", va="center",
                fontsize=subtitle_size)

def arrow(ax, start, end, lw=2.0, color="black"):
    ax.annotate("", xy=end, xytext=start,
                arrowprops=dict(arrowstyle="->", linewidth=lw, color=color,
                                shrinkA=0, shrinkB=0))

## Runtime requirements

This notebook uses:

- `numpy`
- `pandas`
- `matplotlib`
- `scipy`
- `IPython`

In Colab these are usually already available. If running locally, install them with:

```bash
pip install numpy pandas matplotlib scipy ipython
```

## 1. Synthetic design model

The model is intentionally small and interpretable. It provides a reproducible engineering object for studying admissible regions, topology, and robust operating points.

In [ ]:
N = 260
drive = np.linspace(0, 1, N)
loss = np.linspace(0, 1, N)
X, Y = np.meshgrid(drive, loss)
threshold = 0.36

def sigmoid(z, s=0.05):
    return 1 / (1 + np.exp(-z / s))

def support_field(params):
    center = params.get("center", 0.62)
    width = params.get("width", 0.28)
    loss_scale = params.get("loss_scale", 0.20)
    shoulder_amp = params.get("shoulder_amp", 0.35)
    shoulder_x = params.get("shoulder_x", center + 0.18)
    shoulder_y = params.get("shoulder_y", 0.10)
    plateau = sigmoid(X - (center - width), 0.045) * sigmoid((center + width) - X, 0.045)
    low_loss = np.exp(-(Y / loss_scale) ** 1.55)
    shoulder = shoulder_amp * np.exp(-((X - shoulder_x)**2 / 0.025 + (Y - shoulder_y)**2 / 0.020))
    return np.clip(plateau * low_loss + shoulder, 0, 1)

designs = {
    "single cavity": {"center": 0.62, "width": 0.18, "loss_scale": 0.16, "shoulder_amp": 0.25, "cost": 2, "complexity": 2},
    "coupled resonators": {"center": 0.62, "width": 0.22, "loss_scale": 0.18, "shoulder_amp": 0.30, "cost": 4, "complexity": 4},
    "programmable mesh": {"center": 0.60, "width": 0.26, "loss_scale": 0.20, "shoulder_amp": 0.40, "cost": 7, "complexity": 7},
    "hybrid chip": {"center": 0.63, "width": 0.32, "loss_scale": 0.23, "shoulder_amp": 0.42, "cost": 9, "complexity": 8},
}

def region_metrics(params, threshold=threshold):
    Z = support_field(params)
    mask = Z >= threshold
    labels, ncomp = ndimage.label(mask)
    if ncomp == 0:
        return {"Z": Z, "mask": mask, "labels": labels, "ncomp": 0,
                "largest": np.zeros_like(mask, dtype=bool), "dist": np.zeros_like(Z),
                "admitted_area": 0.0, "largest_component": 0.0,
                "max_margin": 0.0, "average_margin": 0.0,
                "x_star": np.nan, "y_star": np.nan}
    sizes = ndimage.sum(mask, labels, index=np.arange(1, ncomp+1))
    largest_id = int(np.argmax(sizes) + 1)
    largest = labels == largest_id
    dist_pixels = ndimage.distance_transform_edt(largest)
    dist_norm = dist_pixels / dist_pixels.max() if dist_pixels.max() > 0 else dist_pixels
    idx = np.unravel_index(np.argmax(dist_pixels), dist_pixels.shape)
    return {
        "Z": Z, "mask": mask, "labels": labels, "ncomp": int(ncomp), "largest": largest,
        "dist": dist_norm, "dist_pixels": dist_pixels,
        "admitted_area": float(mask.mean()),
        "largest_component": float(largest.mean()),
        "max_margin": float(dist_pixels.max()/N),
        "average_margin": float(dist_pixels[largest].mean()/N) if largest.any() else 0.0,
        "x_star": float(drive[idx[1]]), "y_star": float(loss[idx[0]])
    }

alpha, beta, gamma, delta = 1.20, 0.90, 0.020, 0.015
rows, metrics = [], {}
for name, params in designs.items():
    m = region_metrics(params)
    metrics[name] = m
    score = alpha*m["largest_component"] + beta*m["max_margin"] - gamma*params["cost"] - delta*params["complexity"]
    rows.append({
        "design": name, "admitted_area": m["admitted_area"], "largest_component": m["largest_component"],
        "component_count": m["ncomp"], "maximum_margin": m["max_margin"], "average_margin": m["average_margin"],
        "hardware_cost": params["cost"], "control_complexity": params["complexity"], "score": score,
        "x_star": m["x_star"], "y_star": m["y_star"],
    })
df = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
df.to_csv(DATA / "49_v3_architecture_metrics.csv", index=False)
df.round(3)

## 2. The region-constrained design specification

In [ ]:
fig, ax = plt.subplots(figsize=(14.5, 4.2))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
labels = [
    ("Design\n$D$", "candidate\narchitecture"),
    ("Admissible\nregion $\\Omega(D)$", "thresholded\nspecification"),
    ("Largest\ncomponent $\\Omega_c(D)$", "admissible topology\n(leading spec)"),
    ("Distance\ntransform", "$d(\\partial\\Omega_c)$"),
    ("Maximum-margin\npoint $x^*$", "$\\arg\\max d$"),
    ("Fault-tolerant\nexecution", "robust logical\ncomputation"),
]
xs = np.linspace(0.04, 0.86, len(labels))
w, h = 0.12, 0.34
for i, (title, subtitle) in enumerate(labels):
    draw_box(ax, (xs[i],0.48), (w,h), title, subtitle, lw=3.3 if i==2 else 2.0, title_size=10, subtitle_size=8)
    if i < len(labels)-1:
        arrow(ax, (xs[i]+w+0.01,0.65), (xs[i+1]-0.012,0.65), lw=2)
ax.text(0.28,0.22,"SPECIFICATION: determine what is admissible",ha="center",fontsize=9,fontweight="bold")
ax.text(0.72,0.22,"OPTIMIZATION: choose the robust point inside $\\Omega_c$",ha="center",fontsize=9,fontweight="bold")
ax.plot([0.04,0.53],[0.27,0.27],linewidth=1.8)
ax.plot([0.58,0.98],[0.27,0.27],linewidth=1.8)
ax.text(0.5,0.08,"Preserve the region before selecting the operating point.",ha="center",fontsize=12)
ax.set_title("Region-Constrained Design Search Specification")
savefig(fig, "49_v3_01_region_constrained_specification.png")
plt.show()

## 3. Objective-first search vs specification-first search

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Objective-First vs Specification-First Search")
ax.text(0.25,0.88,"Objective-first search",ha="center",fontsize=15,fontweight="bold")
ax.text(0.75,0.88,"Specification-first search",ha="center",fontsize=15,fontweight="bold")
left_steps = ["candidate","score","gradient","repeat"]
for i,s in enumerate(left_steps):
    y=0.70-i*0.15
    draw_box(ax,(0.08,y),(0.34,0.08),s,title_size=10)
    if i<len(left_steps)-1: arrow(ax,(0.25,y),(0.25,y-0.06),lw=2)
right_steps = ["candidate design $D$","$\\Omega(D)$","$\\Omega_c(D)$","$d(\\partial\\Omega_c)$","$x^*$"]
for i,s in enumerate(right_steps):
    y=0.70-i*0.12
    draw_box(ax,(0.60,y),(0.30,0.07),s,title_size=10)
    if i<len(right_steps)-1: arrow(ax,(0.75,y),(0.75,y-0.05),lw=2)
ax.text(0.25,0.12,"can improve a point while missing\nthe admissible topology",ha="center",fontsize=10)
ax.text(0.75,0.12,"preserves the region before\nchoosing the operating point",ha="center",fontsize=10)
ax.text(0.50,0.50,"vs.",ha="center",va="center",fontsize=18,fontweight="bold")
savefig(fig,"49_v3_02_objective_vs_specification_first.png")
plt.show()

## 4. Region objective

In [ ]:
fig, ax = plt.subplots(figsize=(13,5.2))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Region-Constrained Objective Function")
ax.add_patch(Rectangle((0.14,0.68),0.72,0.16,fill=False,linewidth=2.2))
ax.text(0.50,0.76,r"$J(D)=\alpha|\Omega_c(D)|+\beta M(D)-\gamma C(D)-\delta K(D)$",
        ha="center",va="center",fontsize=22)
items=[("+ |$\\Omega_c$|","maximize\nadmissible topology"),("+ M","maximize\nrobustness margin"),("− C","minimize\nhardware cost"),("− K","minimize\ncontrol complexity")]
for x,(t,st) in zip([0.12,0.32,0.56,0.76],items):
    draw_box(ax,(x,0.30),(0.15,0.20),t,st,lw=2.0,title_size=16,subtitle_size=10)
ax.text(0.40,0.14,"maximize region",ha="center",fontsize=12)
arrow(ax,(0.46,0.14),(0.53,0.14),lw=1.8)
ax.text(0.62,0.14,"then choose $x^*$",ha="center",fontsize=12)
savefig(fig,"49_v3_03_region_objective.png")
plt.show()

## 5. Search algorithm overview

In [ ]:
fig, ax = plt.subplots(figsize=(8,6.4))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Region-Constrained Search Algorithm")
steps=[("Generate","candidate designs $D$"),("Evaluate","simulate performance field"),("Threshold","construct $\\Omega(D)$"),("Largest Component","extract $\\Omega_c(D)$"),("Distance Transform","compute $d(\\partial\\Omega_c)$"),("Region Objective","compute $J(D)$"),("Select","$D^*=\\arg\\max J(D)$; $x^*=\\arg\\max d$")]
for i,(t,s) in enumerate(steps):
    y=0.84-i*0.115
    draw_box(ax,(0.20,y),(0.60,0.075),t,s,lw=2.0,title_size=12,subtitle_size=8)
    ax.text(0.15,y+0.038,str(i+1),va="center",ha="center",fontsize=9,fontweight="bold",
            bbox=dict(boxstyle="circle",facecolor="white",edgecolor="black"))
    if i<len(steps)-1: arrow(ax,(0.50,y),(0.50,y-0.035),lw=2.0)
savefig(fig,"49_v3_04_search_algorithm_overview.png")
plt.show()

## 6. Design evolution

In [ ]:
stages=[("initial",{"center":0.62,"width":0.18,"loss_scale":0.16,"shoulder_amp":0.25}),
        ("admitted",{"center":0.62,"width":0.22,"loss_scale":0.18,"shoulder_amp":0.30}),
        ("connected",{"center":0.61,"width":0.27,"loss_scale":0.20,"shoulder_amp":0.36}),
        ("expanded\nlarger admissible topology",{"center":0.63,"width":0.34,"loss_scale":0.23,"shoulder_amp":0.42})]
fig,axes=plt.subplots(1,4,figsize=(14,3.6),sharex=True,sharey=True)
stage_rows=[]
for ax,(title,p) in zip(axes,stages):
    Z=support_field(p); mm=region_metrics(p)
    stage_rows.append({"stage":title.replace("\n"," "),"largest_component":mm["largest_component"],"maximum_margin":mm["max_margin"]})
    ax.imshow(Z,origin="lower",extent=[0,1,0,1],aspect="auto",vmin=0,vmax=1)
    ax.contour(X,Y,Z,levels=[threshold],colors="black",linewidths=1.6)
    ax.scatter([mm["x_star"]],[mm["y_star"]],s=48,zorder=4)
    ax.text(0.05,0.90,f"|Ωc|={mm['largest_component']:.2f}\nM={mm['max_margin']:.2f}",color="white",transform=ax.transAxes,fontsize=9,fontweight="bold")
    ax.set_title(title,fontsize=12); ax.set_xlabel("drive")
axes[0].set_ylabel("loss")
fig.suptitle("Design Evolution: Expand $\\Omega_c$ Before Selecting $x^*$",fontsize=18)
fig.text(0.5,-0.02,"Topology grows before the operating point is chosen.",ha="center",fontsize=12)
savefig(fig,"49_v3_05_design_evolution.png")
plt.show()
stage_df=pd.DataFrame(stage_rows)
stage_df.to_csv(DATA/"49_v3_design_evolution.csv",index=False)
stage_df

## 7. Distance transform selects the operating point

In [ ]:
best_name=df.loc[0,"design"]; best=metrics[best_name]
Z=best["Z"]; largest=best["largest"]; dist=best["dist"]; x_star=best["x_star"]; y_star=best["y_star"]
dist_masked=np.where(largest,dist,np.nan)
fig,ax=plt.subplots(figsize=(9,5.6))
im=ax.imshow(dist_masked,origin="lower",extent=[0,1,0,1],aspect="auto",vmin=0,vmax=1)
ax.contour(X,Y,Z,levels=[threshold],colors="black",linewidths=2.0)
ax.scatter([x_star],[y_star],s=160,zorder=5)
ax.annotate("$x^*$\nmaximum-margin point",xy=(x_star,y_star),xytext=(0.75,0.22),arrowprops=dict(arrowstyle="->",linewidth=2.0),fontsize=12)
ax.text(1.18,0.74,"distance transform\n$d(\\partial\\Omega_c)$\n\n↓\n\narg max\n\n↓\n\n$x^*$",transform=ax.transAxes,ha="center",va="center",fontsize=12)
ax.set_xlabel("drive"); ax.set_ylabel("loss"); ax.set_title("Distance Transform Selects $x^*$")
fig.colorbar(im,ax=ax,label="normalized distance to failure boundary")
savefig(fig,"49_v3_06_distance_transform_selects_xstar.png")
plt.show()

## 8. Failure-boundary geometry

In [ ]:
fig,ax=plt.subplots(figsize=(7,7))
clean_axes(ax); ax.set_xlim(-1.05,1.05); ax.set_ylim(-1.05,1.05); ax.set_aspect("equal")
ax.set_title("Failure Boundary Geometry: Robustness Is the Minimum Distance")
margin_radius=0.62; safe_radius=0.98
ax.scatter([0],[0],s=110,zorder=4)
ax.add_patch(Circle((0,0),margin_radius,fill=False,linestyle="--",linewidth=2.0,edgecolor="red"))
ax.add_patch(Circle((0,0),safe_radius,fill=False,linewidth=1.6,edgecolor="0.65"))
ax.text(0,-0.14,"operating\npoint",ha="center",fontsize=9)
ax.text(0.00,0.66,"active margin set by detector",ha="center",color="red",fontsize=11)
boundaries=[("loss",np.pi,1.00),("timing",0.00,0.95),("calibration",np.pi/2,1.00),("detector",-np.pi/2,margin_radius),("routing",-3*np.pi/4,0.88),("phase",np.pi/4,0.86)]
for name,theta,r in boundaries:
    x,y=r*np.cos(theta),r*np.sin(theta); color="red" if name=="detector" else "black"
    arrow(ax,(0,0),(x,y),lw=2.0,color=color); ax.scatter([x],[y],s=50,color=color,zorder=5)
    ax.text(x*1.12,y*1.12,name,ha="center",va="center",fontsize=10,color=color)
    ax.text(x*0.52,y*0.52,f"{r:.2f}",ha="center",va="center",fontsize=8)
ax.text(0,-1.15,"Large perturbations cross a failure boundary.",ha="center",fontsize=11)
savefig(fig,"49_v3_07_failure_boundary_geometry.png")
plt.show()

## 9. Mathematical algorithm

In [ ]:
fig, ax = plt.subplots(figsize=(10,6.2))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_title("Algorithm: Search Over Regions, Not Points")
pseudo = """for each candidate design D:

    Ω(D)  <- admissible_region(D)

    Ωc(D) <- largest_connected_component(Ω(D))

    M(D)  <- max_distance(Ωc(D), boundary(Ωc(D)))

    J(D)  <- alpha|Ωc(D)| + beta M(D) - gamma C(D) - delta K(D)

D* <- argmax_D J(D)

x* <- argmax_x in Ωc(D*) distance_to_boundary(x)
"""
ax.add_patch(Rectangle((0.08,0.10),0.84,0.75,fill=False,linewidth=2.2))
ax.text(0.13,0.78,pseudo,family="monospace",fontsize=13,va="top")
savefig(fig,"49_v3_08_mathematical_algorithm.png")
plt.show()

## 10. Engineering specification table

In [ ]:
spec=pd.DataFrame([
    ["Design D","candidate architecture or control policy","input to search"],
    ["Ω(D)","admissible region","thresholded specification mask"],
    ["Ωc(D)","largest connected admissible component","connected-component analysis"],
    ["d(∂Ωc)","distance to failure boundary","distance transform"],
    ["x*","maximum-margin operating point","argmax distance inside Ωc"],
    ["J(D)","region objective","weighted engineering specification"],
    ["Fault-tolerant computation","robust logical execution","selected from robust region"],
],columns=["Symbol","Meaning","Computation"])
spec.to_csv(DATA/"49_v3_engineering_specification_table.csv",index=False)
fig,ax=plt.subplots(figsize=(13,4.2))
clean_axes(ax); ax.set_title("Engineering Specification for Region-Constrained Design Search")
tbl=ax.table(cellText=spec.values,colLabels=spec.columns,cellLoc="center",loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.0,1.6)
for (r,c),cell in tbl.get_celld().items():
    cell.set_linewidth(1.0)
    if r==0: cell.set_text_props(fontweight="bold")
savefig(fig,"49_v3_09_engineering_specification_table.png")
plt.show()
spec

## 11. Search progress

In [ ]:
progress=pd.DataFrame({"stage":["initial","admitted","connected","expanded"],"|Ωc| region size":stage_df["largest_component"].to_numpy(),"Margin M":stage_df["maximum_margin"].to_numpy()})
progress.to_csv(DATA/"49_v3_search_progress.csv",index=False)
fig,ax=plt.subplots(figsize=(10,5.2))
ax.plot(progress["stage"],progress["|Ωc| region size"],marker="o",linewidth=2.6,label="|Ωc| region size")
ax.plot(progress["stage"],progress["Margin M"],marker="s",linewidth=2.6,label="Margin M")
ax.fill_between(range(len(progress)),progress["|Ωc| region size"],progress["Margin M"],alpha=0.12)
for i,row in progress.iterrows():
    ax.text(i,row["|Ωc| region size"]+0.01,f"{row['|Ωc| region size']:.2f}",ha="center",fontsize=9)
    ax.text(i,row["Margin M"]+0.01,f"{row['Margin M']:.2f}",ha="center",fontsize=9)
ax.set_ylim(0,max(progress["Margin M"].max(),progress["|Ωc| region size"].max())+0.08)
ax.set_ylabel("relative metric"); ax.set_title("Topology Improves Before Operating Margin")
ax.legend(); ax.grid(alpha=0.25)
savefig(fig,"49_v3_10_search_progress_topology_margin.png")
plt.show()

## 12. Architecture tradeoffs sorted by score

In [ ]:
plot_df=df.copy()
plot_df["region"]=plot_df["largest_component"]/plot_df["largest_component"].max()
plot_df["robustness"]=plot_df["maximum_margin"]/plot_df["maximum_margin"].max()
plot_df["cost"]=plot_df["hardware_cost"]/plot_df["hardware_cost"].max()
plot_df["complexity"]=plot_df["control_complexity"]/plot_df["control_complexity"].max()
plot_df["score_norm"]=(plot_df["score"]-plot_df["score"].min())/(plot_df["score"].max()-plot_df["score"].min())
metrics_to_plot=["region","robustness","cost","complexity","score_norm"]
titles=["|Ωc| topology","robustness margin","hardware cost","control complexity","score"]
fig,axes=plt.subplots(1,len(metrics_to_plot),figsize=(15,4.3),sharey=True)
y=np.arange(len(plot_df))
for ax,col,title in zip(axes,metrics_to_plot,titles):
    ax.barh(y,plot_df[col]); ax.set_title(title,fontsize=11); ax.set_xlim(0,1.05); ax.grid(axis="x",alpha=0.25); ax.set_xlabel("normalized")
axes[0].set_yticks(y); axes[0].set_yticklabels(plot_df["design"]); axes[0].invert_yaxis()
fig.suptitle("Architecture Tradeoffs Sorted by Region-Constrained Score",fontsize=18)
fig.text(0.5,-0.02,"Higher is better for topology, robustness, and score. Lower is better for cost and complexity.",ha="center",fontsize=11)
savefig(fig,"49_v3_11_architecture_tradeoffs_sorted.png")
plt.show()

## 13. Region search pipeline

In [ ]:
fig,ax=plt.subplots(figsize=(15,4.6))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_title("Region Search Pipeline")
steps=[("Physics","what can exist?"),("Available\nResources","what was generated?"),("Admissibility","what survives?"),("Topology\n$\\Omega_c$","largest component"),("Distance\nGeometry","$d(\\partial\\Omega_c)$"),("Region\nObjective","$J(D)$"),("Operating\nPoint","$x^*$"),("Fault-Tolerant\nComputation","execution")]
xs=np.linspace(0.03,0.88,len(steps)); w,h=0.105,0.34
for i,(t,s) in enumerate(steps):
    draw_box(ax,(xs[i],0.40),(w,h),t,s,lw=2.1,title_size=10,subtitle_size=8)
    if i<len(steps)-1: arrow(ax,(xs[i]+w+0.006,0.57),(xs[i+1]-0.006,0.57),lw=1.8,color="0.2")
ax.text(0.5,0.15,"Topology specifies robustness. Robustness specifies operation. Operation specifies computation.",ha="center",fontsize=12)
savefig(fig,"49_v3_12_region_search_pipeline.png")
plt.show()

## 14. Topology is the leading specification

In [ ]:
fig,ax=plt.subplots(figsize=(12,6.2))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_title("Topology Is the Leading Specification")
ax.text(0.28,0.88,"Leading specification",ha="center",fontsize=15,fontweight="bold")
left=["Physics","Available resources","Admissibility","Topology $\\Omega_c$","Robustness","Operation $x^*$"]
for i,item in enumerate(left):
    y=0.76-i*0.105
    draw_box(ax,(0.10,y),(0.36,0.07),item,title_size=10)
    if i<len(left)-1: arrow(ax,(0.28,y),(0.28,y-0.035),lw=1.7)
ax.text(0.75,0.88,"Trailing optimization",ha="center",fontsize=15,fontweight="bold")
right=["Objective","Gradient","Local optimum","Summary metrics"]
for i,item in enumerate(right):
    y=0.68-i*0.13
    draw_box(ax,(0.61,y),(0.28,0.07),item,lw=1.5,title_size=10)
    if i<len(right)-1: arrow(ax,(0.75,y),(0.75,y-0.055),lw=1.6)
ax.text(0.5,0.08,"Region-constrained search treats topology as the specification and objectives as summaries.",ha="center",fontsize=12)
savefig(fig,"49_v3_13_topology_leading_specification.png")
plt.show()

## 15. Symbol summary

In [ ]:
symbol_summary=pd.DataFrame([
    ["D","design","candidate architecture or policy"],
    ["Ω(D)","admissible region","thresholded mask"],
    ["Ωc(D)","largest connected component","connected-component analysis"],
    ["d(∂Ωc)","distance to boundary","distance transform"],
    ["x*","maximum-margin point","argmax inside Ωc"],
    ["J(D)","region objective","α|Ωc| + βM − γC − δK"],
    ["C(D)","hardware cost","estimated cost model"],
    ["K(D)","control complexity","control / calibration model"],
    ["M(D)","robustness margin","maximum distance in Ωc"],
],columns=["Symbol","Meaning","Computation"])
symbol_summary.to_csv(DATA/"49_v3_symbol_summary.csv",index=False)
fig,ax=plt.subplots(figsize=(9,4.7))
clean_axes(ax); ax.set_title("Symbol Summary")
tbl=ax.table(cellText=symbol_summary.values,colLabels=symbol_summary.columns,cellLoc="center",loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1.0,1.45)
for (r,c),cell in tbl.get_celld().items():
    cell.set_linewidth(1.0)
    if r==0: cell.set_text_props(fontweight="bold")
savefig(fig,"49_v3_14_symbol_summary.png")
plt.show()
symbol_summary

## 16. Topology → robustness → operation

In [ ]:
fig,ax=plt.subplots(figsize=(14,4.4))
clean_axes(ax); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_title("Topology → Robustness → Operation")
panel_x=[0.05,0.20,0.35,0.50,0.65,0.80]
titles=["candidate\nfield","threshold","$\\Omega$","$\\Omega_c$","$d(\\partial\\Omega_c)$","$x^*$"]
for i,x in enumerate(panel_x):
    ax.add_patch(Rectangle((x,0.35),0.10,0.25,fill=False,linewidth=1.5))
    ax.text(x+0.05,0.65,titles[i],ha="center",fontsize=9)
ax.imshow(support_field(designs["hybrid chip"]),origin="lower",extent=[panel_x[0],panel_x[0]+0.10,0.35,0.60],aspect="auto",vmin=0,vmax=1)
ax.add_patch(Ellipse((panel_x[1]+0.06,0.43),0.07,0.08,fill=False,linewidth=1.6))
ax.add_patch(Ellipse((panel_x[2]+0.06,0.43),0.07,0.08,fill=False,linewidth=1.6))
ax.add_patch(Ellipse((panel_x[3]+0.06,0.43),0.07,0.08,fill=False,linewidth=2.6))
ax.imshow(np.where(best["largest"],best["dist"],np.nan),origin="lower",extent=[panel_x[4],panel_x[4]+0.10,0.35,0.60],aspect="auto",vmin=0,vmax=1)
ax.imshow(np.where(best["largest"],best["dist"],np.nan),origin="lower",extent=[panel_x[5],panel_x[5]+0.10,0.35,0.60],aspect="auto",vmin=0,vmax=1)
ax.scatter([panel_x[5]+0.062],[0.38],s=30,color="red",zorder=5)
for i in range(len(panel_x)-1):
    arrow(ax,(panel_x[i]+0.11,0.475),(panel_x[i+1]-0.01,0.475),lw=1.8)
ax.text(0.25,0.18,"specify what is admissible",ha="center",fontsize=10,fontweight="bold")
ax.text(0.72,0.18,"optimize within admissible topology",ha="center",fontsize=10,fontweight="bold")
ax.plot([0.05,0.50],[0.22,0.22],linewidth=1.4)
ax.plot([0.55,0.91],[0.22,0.22],linewidth=1.4)
savefig(fig,"49_v3_15_topology_robustness_operation.png")
plt.show()

## 17. Export summary

In [ ]:
summary={
    "notebook":"49_region_constrained_design_search_v3",
    "thesis":"Topology specifies robustness; robustness specifies operation; operation specifies computation.",
    "pipeline":["D","Ω(D)","Ωc(D)","d(∂Ωc)","x*","fault-tolerant execution"],
    "best_design":str(best_name),
    "figure_count":len(list(FIG.glob("*.png"))),
    "data_files":[p.name for p in DATA.glob("*")],
}
Path(DATA/"49_v3_summary.json").write_text(json.dumps(summary,indent=2))
summary

## 18. Package and download notebook outputs

Run this final cell after executing the notebook. It creates a ZIP archive with all generated figures and data.

In Google Colab, the ZIP downloads automatically. In Jupyter, the cell displays a clickable download link.

In [ ]:
# ============================================================
# Package and download Notebook 49 v3.1 outputs
# ============================================================
#
# Run this final cell after executing the notebook.
#
# It creates:
#   49_region_constrained_design_search_v3_outputs.zip
#
# containing:
#   outputs/notebook_49_v3/figures/*.png
#   outputs/notebook_49_v3/data/*.csv
#   outputs/notebook_49_v3/data/*.json
#   README_49_v3_outputs.md
#
# In Google Colab, it will trigger an automatic browser download.
# In Jupyter, it will display a clickable download link.

import shutil
from pathlib import Path
from IPython.display import FileLink, display

package_root = Path("outputs/notebook_49_v3")
figure_dir = package_root / "figures"
data_dir = package_root / "data"

package_root.mkdir(parents=True, exist_ok=True)
figure_dir.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

figure_files = sorted(figure_dir.glob("*.png"))
data_files = sorted([p for p in data_dir.glob("*") if p.is_file()])

manifest = package_root / "README_49_v3_outputs.md"
manifest.write_text(
    "# Notebook 49 v3.1 Outputs\n\n"
    "Generated from `49_region_constrained_design_search_v3_1_full.ipynb`.\n\n"
    "## Contents\n\n"
    "- PNG figures generated by the notebook\n"
    "- CSV / JSON data tables generated by the notebook\n"
    "- This README manifest\n\n"
    "## Figures\n"
    + ("\n".join(f"- figures/{p.name}" for p in figure_files) if figure_files else "- No figures found. Run all cells before packaging.")
    + "\n\n## Data\n"
    + ("\n".join(f"- data/{p.name}" for p in data_files) if data_files else "- No data files found. Run all cells before packaging.")
    + "\n",
    encoding="utf-8",
)

zip_base = Path("49_region_constrained_design_search_v3_outputs")
archive_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=package_root)).resolve()

print(f"Packaged outputs: {archive_path}")
print(f"Figures: {len(figure_files)}")
print(f"Data files: {len(data_files)}")

# Colab auto-download if available; Jupyter fallback otherwise.
try:
    from google.colab import files
    files.download(str(archive_path))
except Exception:
    display(FileLink(str(archive_path)))